<div style="display: flex; flex-direction: column; align-items: center; justify-content: center; text-align: center;">
    
<img src="./image/CESIPATH-removebg-preview.png" alt="Logo ADEME" width="200"/>
<br>    

# 🌿CESIPATH

### Optimisation des tournées de livraison pour la transition écologique
    
<br>

<img src="./image/ADEME-removebg-preview.png" alt="Logo ADEME" width="100"/>
<br>
**Appel à manifestation d'intérêt : Mobilité Multimodale Intelligente**

<br><br><br>

</div>

**Structure :** CesiCDP <br>
**Équipe Projet :** LUU Philippe, AFANE Youcef, RABATEL Antonin, RIVET Alexandre <br>
**Date :** Avril 2026

<br><br>

---


> **Résumé exécutif :** Ce document présente la modélisation mathématique et la résolution algorithmique d'un problème d'optimisation de flotte (VRP). L'objectif est de réduire l'empreinte carbone et les coûts opérationnels des livraisons en milieu urbain, en respectant des contraintes strictes de logistique.

# Informations générales

## Versions du documents : 

Ce tableau retrace toutes les versions du document :

|Version|Date|Auteur(s)|Objet|
|----|----|----|----|
|1.0|02/04/2026|PhilippeLUU, Antonin RABATEL, Alexandre RIVET, Youcef AFANE|Création du document|

## Liste de diffusion :
Ce tableau répertorie tous les destinataires ayant accès au document : 

|Entreprise/Organisme|Destinataire(s)|Contact(s)|Rôle|
|----|----|----|----|
|CesiCDP|LUU Philippe|philippe.luu@cesicdp.fr|Ingénieur|
|CesiCDP|RABATEL Antonin|antonin.rabatel@cesicdp.fr|Ingénieur|
|CesiCDP|RIVET Alexandre|alexandre.rivet@cesicdp.fr|Ingénieur|
|CesiCDP|AFANE Youcef|youcef.afane@cesicdp.fr|Ingénieur|

# Synthèse managériale

# 🚚 Projet ADEME — Optimisation des Tournées de Livraison
## Livrable 1 : Modélisation formelle

**Structure :** CesiCDP  
**Commanditaire :** ADEME — Appel à Manifestation d'Intérêt, Mobilité Multimodale Intelligente  
**Équipe :** RABATEL Antonin, RIVET Alexandre, LUU Philippe, AFANE Youcef  
**Date :** Juin 2025  

---

> *Ce notebook présente la modélisation formelle du problème de tournées de livraison confié à CesiCDP par l'ADEME. Il ne traite pas encore des méthodes de résolution — celles-ci feront l'objet des livrables suivants.*

---

## 1. Contexte et enjeux

### 1.1 La mission ADEME

L'ADEME (Agence de l'Environnement et de la Maîtrise de l'Énergie) a lancé un appel à manifestation d'intérêt pour développer de **nouvelles solutions de mobilité durable** adaptées à différents types de territoires. L'enjeu dépasse la simple optimisation logistique : chaque kilomètre superflu parcouru par un véhicule de livraison représente une émission de CO₂ évitable, une consommation de carburant gaspillée, et une usure inutile des infrastructures routières.

CesiCDP, déjà implantée dans le domaine de la mobilité multimodale intelligente, répond à cet appel avec une solution algorithmique visant à **minimiser la distance totale parcourue** lors des tournées de livraison.

### 1.2 Du cycle eulérien au problème des tournées

Un projet précédent (CesiCDP Grand Est) nous avait amenés à résoudre un problème d'**équipement de lampadaires** : il s'agissait de trouver une tournée passant par *toutes les rues* — un cycle eulérien. Ce problème, polynomial, se résolvait élégamment avec l'algorithme de Hierholzer.

Le problème ADEME est **fondamentalement différent**. L'objectif n'est plus de parcourir toutes les routes, mais de **visiter un ensemble de clients** (villes, entrepôts, points de livraison) en minimisant le coût total du déplacement. Ce changement d'objectif bascule le problème dans une tout autre classe de complexité — nous le démontrerons formellement dans la section dédiée.

| Projet | Objectif | Structure | Complexité |
|--------|----------|-----------|------------|
| Lampadaires (Grand Est) | Parcourir toutes les **arêtes** | Cycle Eulérien | $O(E)$ — Polynomial |
| Livraisons (ADEME) | Visiter tous les **sommets** | VRP / TSP | NP-Complet |

### 1.3 Un problème enrichi de contraintes réelles

Les tournées de livraison en conditions réelles ne se réduisent pas à un simple graphe abstrait. Trois contraintes opérationnelles majeures s'imposent dans le cadre de ce projet :

1. **Restrictions et surcoûts sur les arêtes** — certaines routes sont interdites (travaux, tonnage) ou plus coûteuses (péages, zones réglementées)
2. **Capacité de chargement des véhicules** — chaque véhicule ne peut transporter qu'une quantité limitée de marchandises
3. **Pondération dynamique des routes** — les coûts de trajet varient dans le temps (trafic, météo, restrictions horaires)

Ces contraintes seront formalisées mathématiquement dans les sections suivantes.

## 2. Modélisation formelle

### 2.1 Représentation en graphe

Le réseau de livraison est modélisé par un **graphe non orienté pondéré** :

$$G = (V, E, w)$$

où :
- $V = \{v_0, v_1, \dots, v_{n-1}\}$ est l'ensemble des **sommets**, avec $|V| = n$
  - $v_0$ désigne le **dépôt** (point de départ et de retour de tous les véhicules)
  - $v_1, \dots, v_{n-1}$ désignent les **clients** à livrer
- $E \subseteq \{\{v_i, v_j\} \mid i \neq j\}$ est l'ensemble des **arêtes** représentant les routes existantes
- $w : E \times T \rightarrow \mathbb{R}^+$ est la **fonction de coût dynamique**, associant à chaque arête un coût dépendant du temps $t \in T$

> **Remarque :** Le graphe est *non orienté* — le coût de traversée d'une route est le même dans les deux sens : $w(\{v_i, v_j\}, t) = w(\{v_j, v_i\}, t)$. Cette hypothèse est cohérente avec un réseau routier standard où les distances et temps de trajet sont symétriques.

---

### 2.2 Données associées aux clients

Chaque client $v_i \in V \setminus \{v_0\}$ est caractérisé par :

$$\delta_i \in [1, 20] \subset \mathbb{D}^+ \quad \text{(poids/volume à livrer pour le client } v_i\text{)}$$

---

### 2.3 Modélisation des contraintes

#### Contrainte 1 — Restrictions et surcoûts sur les arêtes

Chaque arête $\{v_i, v_j\} \in E$ est associée à un **statut** :

$$\text{statut}(\{v_i, v_j\}) \in \{\text{libre}, \text{surcoût}, \text{interdit}\}$$

Le coût effectif de traversée est :

$$c(\{v_i, v_j\}) = \begin{cases} +\infty & \text{si } \text{statut}(\{v_i, v_j\}) = \text{interdit} \\ w(\{v_i, v_j\}) \cdot (1 + \tau_{ij}) & \text{si } \text{statut}(\{v_i, v_j\}) = \text{surcoût}, \; \tau_{ij} > 0 \\ w(\{v_i, v_j\}) & \text{sinon} \end{cases}$$

où $\tau_{ij} \in [0, 1]$ est le **taux de surcoût** de l'arête (e.g., $\tau = 0.3$ pour une route à péage majorant le coût de 30 %).

Une arête interdite est équivalente à une absence d'arête dans le graphe résiduel utilisé pour la résolution.

#### Contrainte 2 — Capacité de chargement

Un véhicule unique de capacité maximale $Q \in \mathbb{D}^+$ effectue plusieurs sous-tournées en repartant du dépôt $v_0$ dès que sa charge résiduelle ne lui permet plus de satisfaire le prochain client. Soit $R = (r_1, r_2, \dots, r_m)$ l'ensemble des sous-tournées effectuées, où chaque $r_s = (v_0, v_{i_1}, \dots, v_{i_p}, v_0)$.

Pour toute sous-tournée $r_s$ :

$$\sum_{j=1}^{p} \delta_{i_j} \leq Q$$

#### Contrainte 3 — Pondération dynamique (coûts temporels)

Le coût de traversée d'une arête varie selon l'heure de passage. On discrétise le temps en $|T|$ intervalles horaires :

$$w : E \times T \rightarrow \mathbb{R}^+ \quad \text{avec } T = \{t_0, t_1, \dots, t_m\}$$

Le coût réel au moment $t$ de franchissement de l'arête $\{v_i, v_j\}$ est $w(\{v_i, v_j\}, t)$. En pratique, ce coût peut être modélisé par un coefficient multiplicateur de congestion :

$$w(\{v_i, v_j\}, t) = w_0(\{v_i, v_j\}) \cdot \lambda(t)$$

où $w_0(\{v_i, v_j\})$ est le coût de base et $\lambda(t) \geq 1$ est le **facteur de congestion** à l'instant $t$.

---

### 2.4 Variables de décision

On introduit les variables binaires :

$$y_{ijk} = \begin{cases} 1 & \text{si le véhicule } k \text{ emprunte l'arête } \{v_i, v_j\} \\ 0 & \text{sinon} \end{cases}$$

$$x_{ik} = \begin{cases} 1 & \text{si le client } v_i \text{ est affecté au véhicule } k \\ 0 & \text{sinon} \end{cases}$$

---

### 2.5 Fonction objectif

On cherche à **minimiser le coût total** de l'ensemble des tournées, en tenant compte des coûts dynamiques et des surcoûts d'arêtes :

$$\min \sum_{k=1}^{K} \sum_{\{v_i, v_j\} \in E} y_{ijk} \cdot c\bigl(\{v_i, v_j\}, t_{ijk}\bigr)$$

où $t_{ijk}$ est l'heure à laquelle le véhicule $k$ franchit l'arête $\{v_i, v_j\}$.

---

### 2.6 Formulation synthétique du problème

$$\text{CVRP-DR} \triangleq \begin{cases}
\text{Données :} & G=(V, E, w), \; K \text{ véhicules de capacité } Q, \\
                 & \text{demandes } \delta_i, \; \text{restrictions } \tau_{ij}, \; \text{congestion } \lambda(t) \\
\text{Décision :} & \text{Affecter chaque client à un véhicule, construire } K \text{ tournées} \\
\text{Objectif :} & \text{Minimiser le coût total dynamique des tournées} \\
\text{s.c. :} & \text{(1) Chaque client est visité exactement une fois} \\
              & \text{(2) Chaque tournée respecte la capacité } Q \\
              & \text{(3) Aucune arête interdite n'est empruntée} \\
              & \text{(4) Les coûts reflètent la congestion au moment du passage}
\end{cases}$$

Ce problème est un **Vehicle Routing Problem with Capacity constraints and Dynamic Routing** (CVRP-DR), une généralisation du VRP classique.

## 3. Illustration de la modélisation

Avant d'aborder la théorie de la complexité, illustrons les structures de données sur un exemple jouet : 6 clients, 1 dépôt, 2 véhicules.

In [ ]:
import math
import random

# ─────────────────────────────────────────────────────────────
# Structures de données pour le CVRP-DR
# ─────────────────────────────────────────────────────────────

# Sommets : 0 = dépôt, 1..6 = clients
DEPOT = 0
sommets = list(range(7))  # [0, 1, 2, 3, 4, 5, 6]

# Demandes de chaque client (en unités de charge)
demandes = {
    0: 0,   # dépôt : pas de demande
    1: 10,
    2: 15,
    3: 8,
    4: 20,
    5: 12,
    6: 18,
}

# Capacité de chaque véhicule
CAPACITE = 40
NB_VEHICULES = 2

# Coûts de base entre chaque paire (distance euclidienne sur grille fictive)
# Positions fictives des sommets
positions = {
    0: (0, 0),   # dépôt (centre)
    1: (2, 3),
    2: (-1, 4),
    3: (4, 1),
    4: (3, -2),
    5: (-3, -1),
    6: (-2, 2),
}

def cout_base(u, v):
    """Coût de base entre deux sommets (distance euclidienne)."""
    xu, yu = positions[u]
    xv, yv = positions[v]
    return round(math.sqrt((xu - xv)**2 + (yu - yv)**2), 2)

# ─────────────────────────────────────────────────────────────
# Contrainte 1 : restrictions sur les arêtes
# ─────────────────────────────────────────────────────────────

# Arcs interdits (travaux, poids limite, etc.)
arcs_interdits = {(3, 4), (4, 3)}  # route bidirectionnelle fermée pour travaux

# Surcoûts (péages, zones réglementées) : arc -> taux en [0, 1]
surcoûts = {
    (1, 3): 0.30,  # route à péage : +30 %
    (3, 1): 0.30,
    (0, 4): 0.20,  # zone à faibles émissions : +20 %
    (4, 0): 0.20,
}

def cout_effectif(u, v):
    """
    Coût effectif d'une arête (u, v) en tenant compte des restrictions.
    Retourne float('inf') si l'arête est interdite.
    """
    if (u, v) in arcs_interdits:
        return float('inf')
    base = cout_base(u, v)
    taux = surcoûts.get((u, v), 0.0)
    return round(base * (1 + taux), 2)

# ─────────────────────────────────────────────────────────────
# Contrainte 3 : pondération dynamique
# ─────────────────────────────────────────────────────────────

# Facteurs de congestion par tranche horaire (λ(t))
# Intervalles : matin, heure de pointe matin, journée, heure de pointe soir, soir
TRANCHES_HORAIRES = {
    "matin"           : (6,  8,  1.0),   # (h_debut, h_fin, lambda)
    "pointe_matin"    : (8,  10, 1.8),
    "journee"         : (10, 16, 1.1),
    "pointe_soir"     : (16, 19, 2.0),
    "soir"            : (19, 22, 1.2),
}

def facteur_congestion(heure):
    """Retourne le facteur de congestion λ(t) pour une heure donnée."""
    for nom, (h_debut, h_fin, lam) in TRANCHES_HORAIRES.items():
        if h_debut <= heure < h_fin:
            return lam
    return 1.0  # hors tranches définies : pas de surcharge

def cout_dynamique(u, v, heure):
    """
    Coût réel de franchissement de l'arête (u, v) à l'instant heure.
    Intègre les restrictions statiques et la congestion temporelle.
    """
    c_eff = cout_effectif(u, v)
    if c_eff == float('inf'):
        return float('inf')
    return round(c_eff * facteur_congestion(heure), 2)

# ─────────────────────────────────────────────────────────────
# Affichage synthétique
# ─────────────────────────────────────────────────────────────
print("=== Paramètres du problème ===")
print(f"Sommets : {sommets}  (0 = dépôt)")
print(f"Capacité par véhicule : {CAPACITE}")
print(f"Nombre de véhicules : {NB_VEHICULES}")
print(f"Demande totale : {sum(demandes.values())} / capacité totale : {CAPACITE * NB_VEHICULES}")
print()
print("=== Exemples de coûts ===")
print(f"Arc (0→1) à 9h  : coût base = {cout_base(0, 1)}, coût dynamique = {cout_dynamique(0, 1, 9)}")
print(f"Arc (1→3) à 14h : coût base = {cout_base(1, 3)}, surcoût péage, coût = {cout_dynamique(1, 3, 14)}")
print(f"Arc (3→4) à 11h : interdit (travaux) → coût = {cout_dynamique(3, 4, 11)}")
print(f"Arc (0→2) à 17h : heure de pointe, coût = {cout_dynamique(0, 2, 17)}")

## 4. Analyse de complexité

### 4.1 Rappels sur les classes de complexité

Avant de démontrer la complexité de notre problème, rappelons les définitions fondamentales :

| Classe | Définition |
|--------|------------|
| **P** | Problèmes résolubles en temps *polynomial* par un algorithme déterministe |
| **NP** | Problèmes dont une solution peut être *vérifiée* en temps polynomial |
| **NP-Difficile** | Tout problème de NP se réduit polynomialement à ce problème |
| **NP-Complet** | Problème à la fois dans NP et NP-Difficile |

La question ouverte $P \stackrel{?}{=} NP$ reste l'un des plus grands problèmes non résolus en informatique théorique. La quasi-totalité des experts considère que $P \neq NP$, ce qui signifie que les problèmes NP-Complets n'admettent **aucun algorithme polynomial**.

---

### 4.2 Problème de décision associé

Pour analyser la complexité, on reformule d'abord notre problème d'optimisation en **problème de décision** (réponse binaire oui/non) :

> **CVRP-DR-DEC** : Étant donné un graphe $G = (V, E, w)$, une flotte de $K$ véhicules de capacité $Q$, des demandes $\delta_i$, des restrictions $\tau_{ij}$ et des facteurs de congestion $\lambda(t)$, existe-t-il un ensemble de $K$ tournées réalisables couvrant tous les clients avec un coût total $\leq B$ ?

Résoudre le problème d'optimisation revient à trouver la plus petite valeur de $B$ pour laquelle CVRP-DR-DEC répond **OUI** — autrement dit, la version décisionnelle est au moins aussi difficile que la version optimisation.

---

### 4.3 Le CVRP-DR est dans NP

**Théorème :** CVRP-DR-DEC $\in$ NP.

**Preuve :** Soit un certificat constitué d'un ensemble de $K$ tournées $(r_1, r_2, \dots, r_K)$. L'algorithme de vérification procède en temps polynomial :

1. Vérifier que chaque client apparaît **exactement une fois** parmi toutes les tournées → $O(n)$
2. Pour chaque tournée $r_k$, vérifier que $\sum_i \delta_i \leq Q$ → $O(n)$
3. Pour chaque arc emprunté, vérifier qu'il n'est pas interdit et calculer son coût dynamique → $O(n \cdot |T|)$
4. Vérifier que le coût total $\leq B$ → $O(1)$

Le certificat est de taille polynomiale en $n$ et la vérification est polynomiale. Donc **CVRP-DR-DEC $\in$ NP**. $\square$

---

### 4.4 Le CVRP-DR est NP-Difficile : réduction depuis le Δ-TSP

**Stratégie :** Nous allons montrer que le **Δ-TSP** (Problème du Voyageur de Commerce métrique, connu NP-Complet) se réduit polynomialement au CVRP-DR. Cela prouvera que CVRP-DR est au moins aussi difficile que le Δ-TSP.

**Rappel — le Δ-TSP :** Étant donné un graphe complet pondéré $G' = (V, E', w')$ respectant l'inégalité triangulaire ($w'(u,v) \leq w'(u,z) + w'(z,v)$ pour tout $z$), existe-t-il un cycle hamiltonien de coût $\leq B$ ?

Le Δ-TSP est NP-Complet (Garey & Johnson, 1979 \[1\]) par réduction depuis le Cycle Hamiltonien.

#### Construction de la réduction

Soit une instance du Δ-TSP : graphe complet $G' = (V, E', w')$ avec $|V| = n$, borne $B$.

On construit une instance du CVRP-DR de la façon suivante :

1. **Sommets :** On désigne $v_0$ comme dépôt, les $n-1$ autres comme clients
2. **Demandes :** $\delta_i = 1$ pour tout client $v_i$ (charges unitaires)
3. **Capacité :** $Q = n - 1$ (un seul véhicule peut tout livrer)
4. **Nombre de véhicules :** $K = 1$
5. **Coûts :** $w(v_i, v_j) = w'(v_i, v_j)$ pour tout arc (pas de restriction, congestion nulle : $\tau_{ij} = 0$, $\lambda(t) = 1$ pour tout $t$)
6. **Borne :** $B' = B$

Cette transformation est réalisée en $O(n^2)$, donc **polynomiale**.

#### Preuve de l'équivalence

**($\Rightarrow$)** Si le Δ-TSP répond OUI (il existe un cycle hamiltonien $v_0 \to v_{\sigma(1)} \to \dots \to v_{\sigma(n-1)} \to v_0$ de coût $\leq B$), alors la tournée unique du véhicule dans l'instance CVRP-DR suit ce même cycle. La charge transportée est $\sum \delta_i = n - 1 = Q$ (contrainte respectée), aucune arête n'est interdite, la congestion est neutre : le coût total est $\leq B = B'$. Le CVRP-DR répond **OUI**.

**($\Leftarrow$)** Si le CVRP-DR répond OUI (il existe une tournée de coût $\leq B'$ avec $K=1$ véhicule visitant tous les $n-1$ clients), alors cette tournée constitue un cycle hamiltonien dans $G'$ de coût $\leq B'= B$. Le Δ-TSP répond **OUI**.

**Conclusion :** $\Delta\text{-TSP} \leq_p \text{CVRP-DR}$, donc CVRP-DR est **NP-Difficile**. $\square$

---

### 4.5 Conclusion : NP-Complétude du CVRP-DR

$$\text{CVRP-DR} \in \text{NP} \quad \wedge \quad \text{CVRP-DR est NP-Difficile} \quad \Rightarrow \quad \text{CVRP-DR est NP-Complet}$$

Ce résultat est cohérent avec la littérature : le VRP classique (sans les contraintes dynamiques) est déjà NP-Complet depuis les travaux fondateurs de Dantzig & Ramser (1959) \[2\]. L'ajout de restrictions sur les arêtes et de pondérations dynamiques ne fait qu'accroître la difficulté du problème.

| Étape | Résultat | Méthode |
|-------|----------|---------|
| Modélisation | Le problème = CVRP-DR | Graphe orienté, capacités, coûts dynamiques |
| Appartenance NP | CVRP-DR $\in$ NP | Algorithme certificat en $O(n \cdot |T|)$ |
| NP-Difficulté | CVRP-DR est NP-Difficile | Réduction $\Delta$-TSP $\leq_p$ CVRP-DR en $O(n^2)$ |
| **Conclusion** | **CVRP-DR est NP-Complet** | NP $\cap$ NP-Difficile |

In [ ]:
import itertools

# ─────────────────────────────────────────────────────────────
# Démonstration de la réduction Δ-TSP → CVRP-DR
# ─────────────────────────────────────────────────────────────

def reduction_tsp_vers_cvrp(sommets_tsp, distances_tsp):
    """
    Construit une instance CVRP-DR équivalente à une instance Δ-TSP.
    
    Le véhicule unique a une capacité = n-1 (peut tout livrer),
    les demandes sont unitaires, aucune restriction ni congestion.
    
    Retourne (instance_cvrp, borne_B) où instance_cvrp est un dict
    décrivant le problème CVRP-DR correspondant.
    """
    n = len(sommets_tsp)
    depot = sommets_tsp[0]
    clients = sommets_tsp[1:]

    instance_cvrp = {
        "depot"         : depot,
        "clients"       : clients,
        "demandes"      : {v: 1 for v in clients},   # charges unitaires
        "capacite"      : n - 1,                     # un seul véhicule suffit
        "nb_vehicules"  : 1,
        "distances"     : distances_tsp,             # coûts identiques au TSP
        "arcs_interdits": set(),                     # aucune restriction
        "surtaxes"      : {},                        # aucun surcoût
        "congestion"    : lambda t: 1.0,             # congestion neutre
    }
    borne_B = n  # borne indicative (sera affinée par l'algorithme de résolution)
    return instance_cvrp, borne_B


def cycle_hamiltonien_cout(sommets, distances):
    """
    Calcule le coût du meilleur cycle hamiltonien par force brute.
    Utilisé uniquement pour valider la réduction sur de petits exemples (n ≤ 8).
    Complexité : O((n-1)!) — non utilisable en production.
    """
    depot = sommets[0]
    clients = sommets[1:]
    meilleur = float('inf')
    meilleure_perm = None

    for perm in itertools.permutations(clients):
        tournee = [depot] + list(perm) + [depot]
        cout = sum(
            distances.get(
                (min(tournee[i], tournee[i+1]), max(tournee[i], tournee[i+1])), float('inf')
            )
            for i in range(len(tournee) - 1)
        )
        if cout < meilleur:
            meilleur = cout
            meilleure_perm = tournee

    return meilleur, meilleure_perm


# ── Exemple de validation ──────────────────────────────────────
# Graphe Δ-TSP à 5 sommets (graphe complet, inégalité triangulaire respectée)
sommets_tsp = [0, 1, 2, 3, 4]
distances_tsp = {
    (0, 1): 10, (0, 2): 15, (0, 3): 20, (0, 4): 25,
    (1, 2):  8, (1, 3): 12, (1, 4): 18,
    (2, 3):  9, (2, 4): 14,
    (3, 4):  7,
}

print("=== Validation de la réduction Δ-TSP → CVRP-DR ===")
print(f"Sommets TSP : {sommets_tsp}")
print()

# Résolution du TSP
cout_tsp, cycle_tsp = cycle_hamiltonien_cout(sommets_tsp, distances_tsp)
print(f"[Δ-TSP]   Meilleur cycle hamiltonien : {cycle_tsp}")
print(f"[Δ-TSP]   Coût optimal : {cout_tsp}")
print()

# Construction de l'instance CVRP-DR équivalente
instance_cvrp, borne = reduction_tsp_vers_cvrp(sommets_tsp, distances_tsp)
print("[CVRP-DR] Instance construite :")
print(f"  Dépôt       : {instance_cvrp['depot']}")
print(f"  Clients     : {instance_cvrp['clients']}")
print(f"  Demandes    : {instance_cvrp['demandes']}")
print(f"  Capacité    : {instance_cvrp['capacite']}")
print(f"  Nb véhic.   : {instance_cvrp['nb_vehicules']}")
print()

# Validation : la tournée optimale du CVRP-DR correspond au cycle hamiltonien du TSP
cout_cvrp, tournee_cvrp = cycle_hamiltonien_cout(sommets_tsp, instance_cvrp["distances"])
print(f"[CVRP-DR] Tournée optimale unique : {tournee_cvrp}")
print(f"[CVRP-DR] Coût : {cout_cvrp}")
print()
print(f"Équivalence validée : {cout_tsp == cout_cvrp} (coûts identiques)")
print("=> La réduction est correcte : Δ-TSP ≤p CVRP-DR")

## 5. Conséquences pratiques et choix algorithmiques

### 5.1 Ce que signifie NP-Complet concrètement

La NP-Complétude du CVRP-DR implique (sous l'hypothèse $P \neq NP$) :

- **Aucun algorithme polynomial** ne peut garantir de trouver la solution optimale
- Tout algorithme **exact** a une complexité au moins exponentielle dans le pire cas
- Pour $n$ clients, l'espace des solutions est de l'ordre de $n!$ — absolument impraticable dès $n \geq 20$

À titre indicatif, pour $n = 20$ clients à $10^9$ opérations/seconde :

| Approche | Complexité | Temps estimé |
|----------|-----------|---------------|
| Force brute | $O(n!)$ | $\approx$ 3,9 années |
| Held-Karp (exact) | $O(n^2 \cdot 2^n)$ | $\approx$ 4 secondes |
| Heuristique (plus proche voisin) | $O(n^2)$ | $< 1$ ms |
| Métaheuristique (algorithme génétique) | $O(n^2 \cdot g)$ | quelques ms |

Pour des tournées de livraison réelles impliquant des dizaines à centaines de clients, une **approche heuristique ou métaheuristique** est incontournable.

### 5.2 Stratégie de résolution envisagée

Deux familles d'algorithmes seront explorées dans les prochains livrables :

**Algorithme exact (référence sur petites instances) :**
- Force brute / Held-Karp → garantit l'optimum, utilisable uniquement pour $n \lesssim 15$

**Algorithmes heuristiques/métaheuristiques (instances réelles) :**
- *Plus proche voisin* (greedy) — construction rapide, solution initiale
- *2-opt / 3-opt* — amélioration locale par échange d'arêtes
- *Algorithme génétique ou recuit simulé* — exploration globale de l'espace de solutions

Ces choix seront justifiés et comparés empiriquement dans l'étude expérimentale (livrable final).

---

### 5.3 Impact des contraintes sur la complexité

| Contrainte ajoutée | Impact sur la résolution |
|--------------------|-------------------------|
| Restrictions d'arêtes | Filtrage du graphe avant résolution — $O(|E|)$ de surcoût, négligeable |
| Capacités véhicules | Contrainte de bin packing intégrée → complexité inchangée en ordre, mais espace de solutions plus contraint |
| Pondération dynamique | Recalcul des coûts à chaque décision → $O(|T|)$ de surcoût par arc évalué |

In [ ]:
import math

# ─────────────────────────────────────────────────────────────
# Illustration de l'explosion combinatoire
# ─────────────────────────────────────────────────────────────

OPS_PAR_SECONDE = 1e9  # 10^9 opérations/seconde

print(f"{'n clients':<12} {'Force brute O(n!)':<25} {'Held-Karp O(n²·2ⁿ)':<25} {'Heuristique O(n²)':<20}")
print("-" * 82)

for n in [5, 10, 15, 20, 25, 30]:
    brute     = math.factorial(n) / OPS_PAR_SECONDE
    held_karp = (n**2) * (2**n) / OPS_PAR_SECONDE
    heurist   = (n**2) / OPS_PAR_SECONDE

    def fmt(t):
        if t < 1e-3:   return f"{t*1e6:.2f} µs"
        if t < 1:      return f"{t*1e3:.2f} ms"
        if t < 60:     return f"{t:.2f} s"
        if t < 3600:   return f"{t/60:.1f} min"
        if t < 86400:  return f"{t/3600:.1f} h"
        if t < 3.15e7: return f"{t/86400:.0f} jours"
        return f"{t/3.15e7:.2e} années"

    print(f"{n:<12} {fmt(brute):<25} {fmt(held_karp):<25} {fmt(heurist):<20}")

## 6. Références bibliographiques

\[1\] **Garey, M. R., & Johnson, D. S.** (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W.H. Freeman and Company. — Référence fondatrice sur la NP-complétude, incluant la preuve de NP-complétude du TSP métrique.

\[2\] **Dantzig, G. B., & Ramser, J. H.** (1959). The Truck Dispatching Problem. *Management Science*, 6(1), 80–91. — Article original introduisant le Vehicle Routing Problem.

\[3\] **Toth, P., & Vigo, D.** (Eds.) (2014). *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM. — Référence complète sur les variantes du VRP et leurs algorithmes de résolution.

\[4\] **Christofides, N.** (1976). Worst-case analysis of a new heuristic for the travelling salesman problem. *Technical Report*, Carnegie Mellon University. — Heuristique garantissant une solution à 3/2 de l'optimum pour le Δ-TSP.

\[5\] **Cordeau, J.-F., Laporte, G., Savelsbergh, M. W. P., & Vigo, D.** (2007). Vehicle Routing. In *Handbooks in Operations Research and Management Science*, Vol. 14, pp. 367–428. Elsevier. — Survey complet sur les contraintes de capacité, fenêtres temporelles et dynamisme dans le VRP.

## 7. Synthèse et feuille de route

Ce premier livrable a posé les bases formelles du projet ADEME :

**Ce que nous avons établi :**

1. **Modélisation** — Le problème est formalisé comme un CVRP-DR (Vehicle Routing Problem with Capacity constraints and Dynamic Routing) sur un graphe orienté pondéré, intégrant trois contraintes opérationnelles réalistes : restrictions d'arêtes, capacités de chargement, et pondération dynamique des coûts.

2. **Complexité** — Le CVRP-DR est prouvé NP-Complet par :
   - Appartenance à NP (certificat vérifiable en temps polynomial)
   - NP-difficulté par réduction polynomiale depuis le Δ-TSP ($\Delta\text{-TSP} \leq_p \text{CVRP-DR}$)

3. **Conséquences** — Une approche exacte est inenvisageable au-delà de quelques dizaines de clients. Des algorithmes heuristiques et métaheuristiques seront nécessaires pour les instances réelles.

**Prochaines étapes :**

| Livrable | Contenu |
|----------|---------|
| **Livrable 2** — Conception algorithmique | Implémentation d'au moins deux algorithmes de résolution (exact + heuristique) |
| **Livrable 3** — Étude expérimentale | Génération d'instances aléatoires, benchmarks, analyse statistique des performances |
| **Livrable final** | Consolidation de l'ensemble, conclusions et perspectives |

---

> *Ce notebook respecte les recommandations PEP 8 pour le style de code Python. Les fonctions sont documentées avec des docstrings et les structures de données sont choisies pour leur lisibilité plutôt que leur performance brute, conformément aux objectifs pédagogiques de ce premier livrable.*